# Temporal Analysis Yellow

Data range from 2020 to 2025 (excluding 2025 December)

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f
import findspark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [ ]:
projectRoot = Path.cwd().parents[2]

CLEAN_DATA = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "yellow"
# csv files of plot data
TABLES_DIR = projectRoot / "data" / "plotTables" / "temporal" / "yellow"
os.makedirs(TABLES_DIR, exist_ok=True)


HTML_DIR = projectRoot / "data" / "html" / "temporal" / "yellow"
os.makedirs(HTML_DIR, exist_ok=True)

Some helpful functions

In [ ]:
def save_table(df:pd.DataFrame, name:str):
    path = os.path.join(str(TABLES_DIR), f"{name}.csv")
    df.to_csv(path, index=False)

def load_csv(path:Path) -> pd.DataFrame:
    return pd.read_csv(path)

def make_date_col(df):
    """
    Build a pandas datetime column from pickup_year + pickup_month.

    2020 01 -> 2020-01-01
    """
    df["date"] = pd.to_datetime(
        df["pickup_year"].astype(str) + "-" +
        df["pickup_month"].astype(str) + "-01"
    )
    return df

def add_temporal_columns(df: DataFrame) -> DataFrame:
    """
    Derive useful temporal features.
    """
    df = (
        df
        .withColumn("pickup_hour", f.hour("pickup_datetime"))
        .withColumn("pickup_day_of_week", f.dayofweek("pickup_datetime"))
        .withColumn("pickup_day_name", f.date_format("pickup_datetime", "EEEE"))
        .withColumn("pickup_is_weekend", f.dayofweek("pickup_datetime").isin([1, 7]))
        .withColumn("time_of_day",
            f.when((f.col("pickup_hour") >= 6)  & (f.col("pickup_hour") < 10), "morning_rush")
             .when((f.col("pickup_hour") >= 10) & (f.col("pickup_hour") < 16), "midday")
             .when((f.col("pickup_hour") >= 16) & (f.col("pickup_hour") < 20), "evening_rush")
             .when((f.col("pickup_hour") >= 20) & (f.col("pickup_hour") < 24), "night")
             .otherwise("late_night")
        )
    )
    return df

For Plotly plots formatting

In [ ]:
LAYOUT_DEFAULTS = dict(
    template="plotly_white",
    font=dict(family="Inter, sans-serif", size=12),
    hovermode="x unified",
    margin=dict(l=60, r=30, t=60, b=50),
)

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

COMPANY_COLORS = {
    "Uber": "#000000", "Lyft": "#FF00BF",
    "Via": "#00C48C",  "Juno": "#FFA500",
}

# Spark Init

## No need to run Spark if you run the code from *.csv

`NOTE`:
Change the configs according to your device if you wanna run it (if you don't have the plot tables already in your device).

`If we want to run this code on the cloud,
we will need to change the configs too.`

In [ ]:
findspark.init()

spark = (SparkSession.builder
        .master("local[6]")  
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
spark.sparkContext

In [ ]:
df = spark.read.parquet(str(CLEAN_DATA))
df = add_temporal_columns(df)

# Yearly Trend

In [ ]:
def temp_01_yearly_trend(df:DataFrame, table_path:Path):
    
    if not table_path:
        yearly = (
            df.groupBy("pickup_year")
            .agg(
                f.count("*").alias("total_trips"),
                f.round(f.avg("total_amount"), 2).alias("avg_fare"),
                f.round(f.avg("trip_distance"), 2).alias("avg_miles"),
                f.round(f.avg("trip_time") / 60, 2).alias("avg_duration_min"),
                f.round(f.avg("tip_amount"), 2).alias("avg_tip"),
            )
            .orderBy("pickup_year")
            .toPandas()
        )
        save_table(yearly, "temp_01_yearly_trend")
    else:
        yearly = load_csv(table_path)

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Avg Trip Distance per Year", 
            "Avg Trip Duration per Year", 
            "Average Fare per Year",
            "Avg Tip per Year"
        )
    )

    fig.add_trace(go.Scatter(
        x=yearly["pickup_year"], y=yearly["avg_miles"],
        mode="lines+markers", name="Avg Distance",
        marker=dict(size=8), line=dict(color="#66c2a5", width=2.5),
    ), row=1, col=1)


    fig.add_trace(go.Scatter(
        x=yearly["pickup_year"], y=yearly["avg_duration_min"],
        mode="lines+markers", name="Avg Duration",
        marker=dict(size=8, symbol="square"), line=dict(color="#8da0cb", width=2.5),
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=yearly["pickup_year"], y=yearly["avg_fare"],
        mode="lines+markers", name="Avg Fare",
        marker=dict(size=8), line=dict(color="#fc8d62", width=2.5),
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=yearly["pickup_year"], y=yearly["avg_tip"],
        mode="lines+markers", name="Avg Tip",
        marker=dict(size=8, symbol="square"), line=dict(color="#FA3F1E", width=2.5),
    ), row=2, col=2)

    fig.update_yaxes(title_text="Miles", row=1, col=1)
    fig.update_yaxes(title_text="Minutes", row=1, col=2)
    fig.update_yaxes(title_text="USD ($)", row=2, col=1)
    fig.update_yaxes(title_text="USD ($)", row=2, col=2)

    fig.update_layout(
        title="Yellow - Yearly Overview - (2020 - 2025)",
        showlegend=False, height=900, **LAYOUT_DEFAULTS
    )

    fig.show()
    fig.write_html(HTML_DIR / f"temp_01_yearly_trend.html")

In [ ]:
# temp_01_yearly_trend(df, table_path=None)

temp_01_yearly_trend(None, table_path=TABLES_DIR/"temp_01_yearly_trend.csv")

# Monthly Trips

In [ ]:
def temp_02_monthly_trips(df, table_path):
    """
    Monthly trip volume across years.
    """
    if not table_path:
        monthly = (
            df.groupBy("pickup_year", "pickup_month")
            .agg(f.count("*").alias("total_trips"))
            .orderBy("pickup_year", "pickup_month")
            .toPandas()
        )
        monthly = make_date_col(monthly)
        save_table(monthly, "temp_02_monthly_trips")
    else:
        monthly = load_csv(table_path)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=monthly["date"], y=monthly["total_trips"],
        mode="lines", fill="tozeroy",
        line=dict(color="#8ed6bf", width=2),
        fillcolor="rgba(102,194,165,0.15)",
        name="Trips",
        hovertemplate="%{x|%b %Y}: %{y:,.0f} trips<extra></extra>",
    ))

    # COVID-19
    fig.add_vrect(
        x0="2020-02-01", x1="2020-07-01",
        fillcolor="red", opacity=0.1, line_width=0,
        annotation_text="COVID-19", annotation_position="top left",
        annotation_font_color="red",
    )

    fig.update_layout(
        title="Yellow - Monthly Trip Volume - (2020 - 2025)",
        yaxis_title="Trips", xaxis_title="",
        height=450, **LAYOUT_DEFAULTS,
    )

    fig.show()
    fig.write_html(HTML_DIR / f"temp_02_monthly_trips.html")

In [ ]:
#temp_02_monthly_trips(df, None)

temp_02_monthly_trips(None, TABLES_DIR / "temp_02_monthly_trips.csv")

# Day of Week

In [ ]:
def temp_03_day_of_week(df:DataFrame, table_path:Path):
    """Trip, volume and avg fare by day of week."""
    if not table_path:
        dow = (
            df
            .groupBy("pickup_day_name", "pickup_day_of_week")
            .agg(
                f.count("*").alias("total_trips"),
                f.round(f.avg("total_amount"), 2).alias("avg_fare"),
            )
            .orderBy("pickup_day_of_week")
            .toPandas()
        )
        dow["pickup_day_name"] = pd.Categorical(
            dow["pickup_day_name"], categories=DAY_ORDER, ordered=True
        )
        dow = dow.sort_values("pickup_day_name")
        dow["color"] = dow["pickup_day_name"].apply(
            lambda d: "#F0A9B8" if d in ("Saturday", "Sunday") else "#8fa7c9"
        )
        save_table(dow, "temp_03_day_of_week")
    else:
        dow = load_csv(table_path)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Total Trips by Day of Week", "Avg Fare by Day of Week"),
    )

    fig.add_trace(go.Bar(
        x=dow["pickup_day_name"], y=dow["total_trips"],
        marker_color=dow["color"], name="Trips",
        text=dow["total_trips"].apply(lambda x: f"{x/1e6:.1f}M"),
        textposition="outside",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=dow["pickup_day_name"], y=dow["avg_fare"],
        marker_color=dow["color"], name="Avg Fare",
        text=dow["avg_fare"].apply(lambda x: f"${x:.2f}"),
        textposition="outside",
    ), row=1, col=2)

    fig.update_yaxes(title_text="Trips", row=1, col=1)
    fig.update_yaxes(title_text="USD ($)", row=1, col=2)

    fig.update_layout(
        title="Yellow — Day of Week Patterns",
        showlegend=False, height=450, **LAYOUT_DEFAULTS,
    )
    fig.show()
    fig.write_html(HTML_DIR / f"temp_03_day_of_week.html")

In [ ]:
# temp_03_day_of_week(df, None)

temp_03_day_of_week(None, TABLES_DIR / "temp_03_day_of_week.csv")

# Hourly Demand

In [ ]:
def temp_04_hourly_demand(df:DataFrame, table_path:Path):
    """
    Hourly demand: weekday vs weekend
    """
    if not table_path:
        hourly = (
            df
            .groupBy("pickup_hour", "pickup_is_weekend")
            .agg(f.count("*").alias("total_trips"))
            .orderBy("pickup_hour")
            .toPandas()
        )
        hourly["day_type"] = hourly["pickup_is_weekend"].map(
            {True: "Weekend", False: "Weekday"}
        )
        save_table(hourly, "temp_04_hourly_demand")
    else:
        hourly = load_csv(table_path)

    fig = go.Figure()

    for day_type, color in [("Weekday", "#66c2a5"), ("Weekend", "#e78ac3")]:
        sub = hourly[hourly["day_type"] == day_type].sort_values("pickup_hour")
        fig.add_trace(go.Scatter(
            x=sub["pickup_hour"], y=sub["total_trips"],
            mode="lines+markers", name=day_type,
            line=dict(color=color, width=2.5),
            marker=dict(size=6),
            hovertemplate="%{x}:00 — %{y:,.0f} trips<extra></extra>",
        ))

    # Rush hour shading
    fig.add_vrect(x0=6, x1=10, fillcolor="orange", opacity=0.07, line_width=0,
                  annotation_text="AM Rush", annotation_position="top left",
                  annotation_font_size=10, annotation_font_color="orange")
    fig.add_vrect(x0=16, x1=20, fillcolor="red", opacity=0.07, line_width=0,
                  annotation_text="PM Rush", annotation_position="top left",
                  annotation_font_size=10, annotation_font_color="red")

    fig.update_layout(
        title="Yellow - Hourly Demand: Weekday vs Weekend (2020 - 2025)",
        xaxis=dict(title="Hour of Day", tickmode="linear", dtick=1,
                   ticktext=[f"{h:02d}:00" for h in range(24)],
                   tickvals=list(range(24))),
        yaxis_title="Total Trips",
        height=500, **LAYOUT_DEFAULTS,
    )
    fig.show()
    fig.write_html(HTML_DIR / f"temp_04_hourly_demand.html")

In [ ]:
# temp_04_hourly_demand(df, None)

temp_04_hourly_demand(None, TABLES_DIR / "temp_04_hourly_demand.csv")

# Time of Day

In [ ]:
def temp_05_time_of_day(df: DataFrame, table_path:Path):
    """Trips, avg fare, avg wait by time-of-day bucket."""
    if not table_path:
        tod = (
            df
            .groupBy("time_of_day")
            .agg(
                f.count("*").alias("total_trips"),
                f.round(f.avg("total_amount"), 2).alias("avg_fare"),
            )
            .toPandas()
        )
        bucket_order = ["late_night", "morning_rush", "midday", "evening_rush", "night"]
        bucket_labels = ["Late Night\n(0-5h)", "Morning Rush\n(6-9h)", "Midday\n(10-15h)",
                        "Evening Rush\n(16-19h)", "Night\n(20-23h)"]
        
        tod["time_of_day"] = pd.Categorical(tod["time_of_day"], categories=bucket_order, ordered=True)
        tod = tod.sort_values("time_of_day")
        tod["label"] = bucket_labels
        save_table(tod, "temp_05_time_of_day")
    else:
        tod = load_csv(table_path)

    colors = ["#91abe5", "#f1a688", "#7DD5B9", "#ffc3e8", "#bbd491"]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Trips by Time of Day", "Avg Fare by Time of Day", "Avg Wait Time"),
    )

    fig.add_trace(go.Bar(
        x=tod["label"], y=tod["total_trips"], marker_color=colors, name="Trips",
        text=tod["total_trips"].apply(lambda x: f"{x/1e6:.1f}M"), textposition="outside",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=tod["label"], y=tod["avg_fare"], marker_color=colors, name="Fare",
        text=tod["avg_fare"].apply(lambda x: f"${x:.2f}"), textposition="outside",
    ), row=1, col=2)

    fig.update_yaxes(title_text="Trips", row=1, col=1)
    fig.update_yaxes(title_text="USD ($)", row=1, col=2)

    fig.update_layout(
        title="Yellow — Time of Day Analysis",
        showlegend=False, height=450, **LAYOUT_DEFAULTS,
    )
    
    fig.show()
    fig.write_html(HTML_DIR / f"temp_05_time_of_day.html")

In [ ]:
# temp_05_time_of_day(df, None)

temp_05_time_of_day(None, TABLES_DIR / "temp_05_time_of_day.csv")

# Hour x Day of Week Heatmap

In [ ]:
def temp_06_heatmap(df:DataFrame, table_path:Path):
    """Heatmap: hour x day_of_week."""
    
    if not table_path:

        heat = (
            df.groupBy("pickup_day_name", "pickup_hour")
            .agg(f.count("*").alias("trips"))
            .toPandas()
        )
        heat["pickup_day_name"] = pd.Categorical(
            heat["pickup_day_name"], categories=DAY_ORDER, ordered=True
        )
        save_table(heat, "temp_06_heatmap")
    else:
        heat = load_csv(table_path)
    
    pivot = heat.pivot_table(
            index="pickup_day_name", columns="pickup_hour",
            values="trips", aggfunc="sum"
        ).reindex(DAY_ORDER)

    fig = go.Figure(data=go.Heatmap(
        z=pivot.values / 1e6,
        x=[f"{h:02d}:00" for h in pivot.columns],
        y=pivot.index.tolist(),
        colorscale="RdBu_r",
        colorbar=dict(title="Trips (M)"),
        text=np.round(pivot.values / 1e6, 1),
        texttemplate="%{text:.1f}",
        textfont=dict(size=9),
        hovertemplate="%{y}, %{x}<br>%{z:.2f}M trips<extra></extra>",
    ))

    fig.update_layout(
        title="Yellow — Trip Heatmap: Day x Hour (Millions)",
        xaxis_title="Hour of Day",
        yaxis=dict(autorange="reversed"),
        height=450, width=1000, **LAYOUT_DEFAULTS,
    )
    fig.show()
    fig.write_html(HTML_DIR / f"temp_06_heatmap.html")

In [ ]:
# temp_06_heatmap(df, None)

temp_06_heatmap(None, TABLES_DIR / "temp_06_heatmap.csv")

# Borough

In [ ]:
def temp_09_borough_temporal(df: DataFrame, path_file: Path):
    """All pickup boroughs: monthly trip trends."""
    if not path_file:
        boro = (
            df.groupBy("pickup_year", "pickup_month", "pickup_borough")
            .agg(f.count("*").alias("trips"))
            .toPandas()
        )
        boro = make_date_col(boro)
        
        save_table(boro, "temp_09_borough_temporal")
    else:
        boro = load_csv(path_file)
    all_boroughs = boro["pickup_borough"].unique()
    borough_colors = px.colors.qualitative.Set2

    fig = go.Figure()
    for i, borough in enumerate(all_boroughs):
        sub = boro[boro["pickup_borough"] == borough].sort_values("date")
        fig.add_trace(go.Scatter(
            x=sub["date"], y=sub["trips"],
            mode="lines", name=borough,
            line=dict(color=borough_colors[i % len(borough_colors)], width=2),
            hovertemplate=f"{borough}<br>" + "%{x|%b %Y}: %{y:,.0f}<extra></extra>",
        ))

    fig.update_layout(
        title="Yellow - Monthly Trips by Borough (2020- 2025)",
        yaxis_title="Trips", xaxis_title="",
        height=500, **LAYOUT_DEFAULTS,
    )
    fig.show()
    fig.write_html(HTML_DIR / f"temp_09_borough_temporal.html")

In [ ]:
# temp_09_borough_temporal(df, None)

temp_09_borough_temporal(None, TABLES_DIR / "temp_09_borough_temporal.csv")

# Yearly Total trips

In [ ]:
def temp_10_yearly_trips(df:DataFrame, table_path:Path):
    if not table_path:
        yearly = (
        df.groupBy("pickup_year")
        .agg(f.count("*").alias("total_trips"))
        .orderBy("pickup_year")
        .toPandas()
        )
        save_table(yearly, "temp_10_yearly_trips")
    else:
        yearly = load_csv(table_path)
    
    fig = go.Figure(go.Scatter(
    x=yearly["pickup_year"],
    y=yearly["total_trips"],
    mode='lines+markers',
    line=dict(width=3, color='#2E86AB'),
    marker=dict(size=10, color='#A23B72'),
    hovertemplate="<b>Year:</b> %{x}<br>" +
                "<b>Total Trips:</b> %{y:,}<br>" +
                "<extra></extra>"
    ))

    fig.update_layout(
        title="Yellow - Total Trips - (2020 - 2025)",
        xaxis_title="Year",
        yaxis_title="Trips",
        height=450, **LAYOUT_DEFAULTS
    )

    fig.show()
    fig.write_html(HTML_DIR / f"temp_10_yearly_trips.html")

In [ ]:
# temp_10_yearly_trips(df, None)

temp_10_yearly_trips(None, TABLES_DIR / "temp_10_yearly_trips.csv")

# Monthly Trends

In [ ]:
def temp_11_monthly_trips(df:DataFrame, table_path:Path):
    if not table_path:
        monthly = (
        df.groupBy("pickup_month")
        .agg(f.count("*").alias("total_trips"))
        .orderBy("pickup_month")
        .toPandas()
        )
        save_table(monthly, "temp_11_monthly_trips")
    else:
        monthly = load_csv(table_path)
    
    fig = go.Figure(go.Bar(
    x=monthly["pickup_month"],
    y=monthly["total_trips"],
    marker_color="#66c2a5",
    hovertemplate="<b>Year:</b> %{x}<br>" +
                "<b>Total Trips:</b> %{y:,}<br>" +
                "<extra></extra>"
    ))

    fig.update_layout(
        title="Yellow - Total Trips - (2020 - 2025)",
        xaxis_title="Month",
        yaxis_title="Trips",
        height=450, **LAYOUT_DEFAULTS
    )

    fig.show()

In [ ]:
# temp_11_monthly_trips(df, None)

# temp_11_monthly_trips(None, TABLES_DIR / "temp_11_monthly_trips.csv")

In [ ]:
spark.stop()